# Retail Sales Analytics: Week 1 Data Cleaning

## Business Problem
Analyse retail sales performance, customer purchasing behavior and product performance to identify trends, opportunities and areas requiring management attention.

Key Question:
- How much revenue and profit does the business generate?
- Which products, categories and regions perform best?
- Which customers contribute the most revenue?
- How are sales changing over time?
- Which products or areas need attention?

## 1. Loading the Data

In [1]:
import pandas as pd
import numpy as np

raw = pd.read_csv("../01_Raw_Data/superstore.csv", encoding="latin-1")
df = raw.copy()
print(df.shape)

(9994, 21)


## 2. Inspecting the dataset

In [2]:
print(df.columns.tolist())
print(df.dtypes)
df.head()


['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']
Row ID             int64
Order ID             str
Order Date           str
Ship Date            str
Ship Mode            str
Customer ID          str
Customer Name        str
Segment              str
Country              str
City                 str
State                str
Postal Code        int64
Region               str
Product ID           str
Category             str
Sub-Category         str
Product Name         str
Sales            float64
Quantity           int64
Discount         float64
Profit           float64
dtype: object


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [3]:
#the dataset profile
print("Rows:", df.shape[0], "| Columns:", df.shape[1])
print("Date range:", pd.to_datetime(df["Order Date"], format="%m/%d/%Y").min().date(),
      "to", pd.to_datetime(df["Order Date"], format="%m/%d/%Y").max().date())
print("Customers:", df["Customer ID"].nunique())
print("Orders:", df["Order ID"].nunique())
print("Products (by ID):", df["Product ID"].nunique())
print("Products (by name):", df["Product Name"].nunique())
print("Categories:", df["Category"].nunique())
print("Sub-categories:", df["Sub-Category"].nunique())
print("Regions:", df["Region"].nunique())

Rows: 9994 | Columns: 21
Date range: 2014-01-03 to 2017-12-30
Customers: 793
Orders: 5009
Products (by ID): 1862
Products (by name): 1850
Categories: 3
Sub-categories: 17
Regions: 4


The dataset has 9994 rows and 21 columns covering orders from January 2014 to December 2017, from 793 customers across 4 regions.

## 3. Data cleaning
### 3.1 Standardizing column names and data types
Column names converted to lowercase with underscores. Dates parsed with an explicit month/day/year format so a wrong format raises an error instead of silently creating blanks. Postal codes converted to 5-character text because numeric storage dropped leading zeros.


In [4]:
df.columns = (df.columns.str.strip().str.lower()
              .str.replace(r"[^a-z0-9]+", "_", regex=True).str.strip("_"))
df["order_date"] = pd.to_datetime(df["order_date"], format="%m/%d/%Y")
df["ship_date"]  = pd.to_datetime(df["ship_date"],  format="%m/%d/%Y")
df["postal_code"] = df["postal_code"].astype(str).str.zfill(5)
df["quantity"] = df["quantity"].astype(int)

print(df.dtypes)


row_id                    int64
order_id                    str
order_date       datetime64[us]
ship_date        datetime64[us]
ship_mode                   str
customer_id                 str
customer_name               str
segment                     str
country                     str
city                        str
state                       str
postal_code                 str
region                      str
product_id                  str
category                    str
sub_category                str
product_name                str
sales                   float64
quantity                  int64
discount                float64
profit                  float64
dtype: object


### 3.2 Missing values an text consistency

In [5]:
text_cols = ["order_id","ship_mode","customer_id","customer_name","segment","country",
             "city","state","region","product_id","category","sub_category","product_name"]
print("Missing values in total:", df.isna().sum().sum())
print("Blank strings:", (df[text_cols].apply(lambda s: s.str.strip() == "")).sum().sum())

for c in text_cols:
    n = (df[c] != df[c].str.strip()).sum()
    if n:
        print(c, "->", n, "values had extra spaces")
    df[c] = df[c].str.strip()

for c in ["category","sub_category","region","segment","ship_mode"]:
    print(df[c].value_counts(), "\n")


Missing values in total: 0
Blank strings: 0
product_name -> 16 values had extra spaces
category
Office Supplies    6026
Furniture          2121
Technology         1847
Name: count, dtype: int64 

sub_category
Binders        1523
Paper          1370
Furnishings     957
Phones          889
Storage         846
Art             796
Accessories     775
Chairs          617
Appliances      466
Labels          364
Tables          319
Envelopes       254
Bookcases       228
Fasteners       217
Supplies        190
Machines        115
Copiers          68
Name: count, dtype: int64 

region
West       3203
East       2848
Central    2323
South      1620
Name: count, dtype: int64 

segment
Consumer       5191
Corporate      3020
Home Office    1783
Name: count, dtype: int64 

ship_mode
Standard Class    5968
Second Class      1945
First Class       1538
Same Day           543
Name: count, dtype: int64 



### 3.3 Logical and relationship checks

In [6]:
print("Sales <= 0:", (df.sales <= 0).sum())
print("Quantity <= 0:", (df.quantity <= 0).sum())
print("Discount outside 0-1:", ((df.discount < 0) | (df.discount > 1)).sum())
print("Ship date before order date:", (df.ship_date < df.order_date).sum())
print("Customer IDs with >1 name:", (df.groupby("customer_id")["customer_name"].nunique() > 1).sum())
print("Product IDs with >1 name:", (df.groupby("product_id")["product_name"].nunique() > 1).sum())

Sales <= 0: 0
Quantity <= 0: 0
Discount outside 0-1: 0
Ship date before order date: 0
Customer IDs with >1 name: 0
Product IDs with >1 name: 32


32 Product IDs are shared by more than one product name. I kept them because they are different products. Product analysis will use product name instead of product ID.

### 3.4 Duplicates
Row ID is unique per row, so it is excluded from the duplicate check.

In [7]:
cols = [c for c in df.columns if c != "row_id"]
print("Duplicates ignoring row_id:", df.duplicated(subset=cols).sum())
display(df[df.duplicated(subset=cols, keep=False)])

df = df.drop_duplicates(subset=cols).reset_index(drop=True)
print("Rows after removing duplicates:", df.shape[0])

Duplicates ignoring row_id: 1


,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
3405,3406,US-2014-150119,2014-04-23,2014-04-27,Standard Class,LB-16795,Laurel Beltran,Home Office,United States,Columbus,...,43229,East,FUR-CH-10002965,Furniture,Chairs,Global Leather Highback Executive Chair with P...,281.372,2,0.3,-12.0588
3406,3407,US-2014-150119,2014-04-23,2014-04-27,Standard Class,LB-16795,Laurel Beltran,Home Office,United States,Columbus,...,43229,East,FUR-CH-10002965,Furniture,Chairs,Global Leather Highback Executive Chair with P...,281.372,2,0.3,-12.0588


Rows after removing duplicates: 9993


### 3.5 Outliers

In [8]:
display(df.nlargest(5, "sales")[["order_id","product_name","sales","profit"]])
display(df.nsmallest(5, "profit")[["order_id","product_name","discount","profit"]])
print("Rows with negative profit:", (df.profit < 0).sum())

,order_id,product_name,sales,profit
2697,CA-2014-145317,Cisco TelePresence System EX90 Videoconferenci...,22638.480,-1811.0784
6825,CA-2016-118689,Canon imageCLASS 2200 Advanced Copier,17499.950,8399.9760
8152,CA-2017-140151,Canon imageCLASS 2200 Advanced Copier,13999.960,6719.9808
2623,CA-2017-127180,Canon imageCLASS 2200 Advanced Copier,11199.968,3919.9888
4189,CA-2017-166709,Canon imageCLASS 2200 Advanced Copier,10499.970,5039.9856


,order_id,product_name,discount,profit
7771,CA-2016-108196,Cubify CubeX 3D Printer Double Head Print,0.7,-6599.9780
683,US-2017-168116,Cubify CubeX 3D Printer Triple Head Print,0.5,-3839.9904
9773,CA-2014-169019,GBC DocuBind P400 Electric Binding System,0.8,-3701.8928
3011,CA-2017-134845,Lexmark MX611dhe Monochrome Laser Printer,0.7,-3399.9800
4990,US-2017-122714,Ibico EPK-21 Electric Binding System,0.8,-2929.4845


Rows with negative profit: 1870


Very large sales and large negative profits are real business events (big orders, heavy discounts), so I kept them. Negative profit is a finding to analyse, not an error.

## 4. Calculated Fields
- year, month, month_name: extracted from order_date
- year_month: label such as 2016-Nov
- year_month_sort: 2016-11, sorts correctly
- ship_days: ship_date minus order_date
- profit_margin_pct: profit / sales x 100
- is_loss: True when profit is negative
- Average Order Value is calculated at report level: total sales / unique orders

In [9]:
df["year"] = df["order_date"].dt.year
df["month"] = df["order_date"].dt.month
df["month_name"] = df["order_date"].dt.strftime("%b")
df["year_month"] = df["order_date"].dt.strftime("%Y-%b")
df["year_month_sort"] = df["order_date"].dt.strftime("%Y-%m")
df["ship_days"] = (df["ship_date"] - df["order_date"]).dt.days
df["profit_margin_pct"] = df["profit"] / df["sales"] * 100
df["is_loss"] = df["profit"] < 0

print("Average Order Value:", round(df.sales.sum() / df.order_id.nunique(), 2))
df.head()

Average Order Value: 458.56


,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,discount,profit,year,month,month_name,year_month,year_month_sort,ship_days,profit_margin_pct,is_loss
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,0.00,41.9136,2016,11,Nov,2016-Nov,2016-11,3,16.00,False
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,0.00,219.5820,2016,11,Nov,2016-Nov,2016-11,3,30.00,False
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,0.00,6.8714,2016,6,Jun,2016-Jun,2016-06,4,47.00,False
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,0.45,-383.0310,2015,10,Oct,2015-Oct,2015-10,7,-40.00,True
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,0.20,2.5164,2015,10,Oct,2015-Oct,2015-10,7,11.25,False


## 5. Saving the cleaned dataset

In [10]:
df.to_csv("../02_Cleaned_Data/superstore_cleaned.csv", index=False)
df.to_excel("../02_Cleaned_Data/superstore_cleaned.xlsx", index=False)

check = pd.read_csv("../02_Cleaned_Data/superstore_cleaned.csv", parse_dates=["order_date","ship_date"])
print(check.shape, "| order_date type:", check["order_date"].dtype)

(9993, 29) | order_date type: datetime64[us]


## 6. Data dictionary

In [11]:
desc = {
 "row_id": "Row number from the source file",
 "order_id": "Unique order identifier",
 "order_date": "Date the order was placed",
 "ship_date": "Date the order was shipped",
 "ship_mode": "Shipping method",
 "customer_id": "Unique customer identifier",
 "customer_name": "Customer name",
 "segment": "Customer segment (Consumer, Corporate, Home Office)",
 "country": "Customer country",
 "city": "Customer city",
 "state": "Customer state",
 "postal_code": "Postal code (5-character text)",
 "region": "Sales region",
 "product_id": "Product identifier (not unique per product)",
 "category": "Product category",
 "sub_category": "Product sub-category",
 "product_name": "Product name",
 "sales": "Revenue from the order line (USD)",
 "quantity": "Units sold",
 "discount": "Discount applied (0 to 1)",
 "profit": "Profit on the order line (negative = loss)",
 "year": "Year of order_date",
 "month": "Month number of order_date",
 "month_name": "Abbreviated month name",
 "year_month": "Year and month label, e.g. 2016-Nov",
 "year_month_sort": "Year-month for correct sorting, e.g. 2016-11",
 "ship_days": "Days between order and shipping",
 "profit_margin_pct": "Profit divided by sales times 100",
 "is_loss": "True when profit is negative",
}
dd = pd.DataFrame({"column": df.columns, "data_type": df.dtypes.astype(str).values})
dd["description"] = dd["column"].map(desc)
print("Columns without description:", dd["description"].isna().sum())   # must be 0
dd.to_csv("../02_Cleaned_Data/data_dictionary.csv", index=False)
dd

Columns without description: 0


,column,data_type,description
0,row_id,int64,Row number from the source file
1,order_id,str,Unique order identifier
2,order_date,datetime64[us],Date the order was placed
3,ship_date,datetime64[us],Date the order was shipped
4,ship_mode,str,Shipping method
5,customer_id,str,Unique customer identifier
6,customer_name,str,Customer name
7,segment,str,"Customer segment (Consumer, Corporate, Home Of..."
8,country,str,Customer country
9,city,str,Customer city


**Cleaning summary**
| Issue | Finding | Action | Reason |
|---|---|---|---|
| Encoding | UTF-8 failed | Read with latin-1 | File contains non-UTF-8 characters |
| Date format | Text, month/day/year | Converted to dates with explicit format | Needed for time analysis |
| Postal Code | Stored as numbers, 449 lost leading zero | Converted to 5-character text | Preserves real codes |
| Missing values | None | None needed | |
| Whitespace | 16 product names had trailing spaces | Trimmed | Avoids false mismatches |
| Duplicates | 1 exact duplicate ignoring row_id | Removed (9,994 to 9,993 rows) | Prevents double counting |
| Product ID | 32 IDs map to more than one name | Kept; analyse by product name | Different products share IDs |
| Outliers | Large sales and large losses | Kept | Genuine business events |
| Date logic | No ship date before order date | None needed | |